# PyTorch Model Optimization & Hyperparameter Tuning

Systematic optimization of facial emotion recognition models through hyperparameter search, architecture variations, and ensemble methods.

**Optimization Strategies:**
1. **Hyperparameter Grid Search** — Learning rate, batch size, weight decay, dropout
2. **Architecture Variations** — Different CNN depths, widths, and regularization
3. **Optimizer Comparison** — Adam, SGD with momentum, AdamW
4. **Learning Rate Scheduling** — Cosine annealing, polynomial decay, warmup
5. **Ensemble Methods** — Combining multiple models for better predictions
6. **Loss Functions** — Focal loss for class imbalance

**Objectives:**
- Improve upon current best (ResNet 62.80%)
- Identify optimal hyperparameter combinations
- Create ensemble of top models
- Achieve 65%+ test accuracy if possible
- Document all experiments and findings

## 1. Setup and Configuration

In [7]:
import os
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from itertools import product
torch.cuda.empty_cache()
# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Clear cached imports
for module in list(sys.modules.keys()):
    if module.startswith('src'):
        del sys.modules[module]

# Import utilities
from src.pytorch_models import get_model
from src.pytorch_train import train_model, create_dataloaders
from src.pytorch_evaluate import create_evaluation_report

# Set seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Configure device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Load data
data_dir = os.path.join('..', 'data', 'preprocessed')
X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
X_val = np.load(os.path.join(data_dir, 'X_val.npy'))
X_test = np.load(os.path.join(data_dir, 'X_test.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
y_val = np.load(os.path.join(data_dir, 'y_val.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))

# Load class weights
with open(os.path.join(data_dir, 'class_weights.json'), 'r') as f:
    class_weights_dict = json.load(f)
class_weights_list = [class_weights_dict[str(i)] for i in range(7)]
class_weights = torch.tensor(class_weights_list, dtype=torch.float32, device=device)

print(f"\n✓ Data loaded: {X_train.shape[0]} train, {X_val.shape[0]} val, {X_test.shape[0]} test")
print(f"✓ Class weights: {[f'{w:.2f}' for w in class_weights_list]}")

Device: cuda
GPU Available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU Memory: 4.29 GB

✓ Data loaded: 22967 train, 5742 val, 7178 test
✓ Class weights: ['1.03', '9.40', '1.00', '0.57', '0.83', '0.85', '1.29']


## 2. Define Hyperparameter Grid Search

In [8]:
# Define comprehensive hyperparameter grid
hyperparameter_grid = {
    'architecture': ['baseline', 'advanced', 'resnet', 'seresnet', 'densenet', 'inception', 'mobilenetv2'],
    'learning_rate': [5e-5, 1e-4, 5e-4, 1e-3],
    'batch_size': [16, 32, 48],
    'weight_decay': [1e-5, 1e-4, 1e-3],
    'dropout': [0.3, 0.5],
}

# Create experiment configurations (sample for initial run)
# For full grid: ~7*4*3*3*2 = 504 configurations (too many for demo)
# Sample key combinations for efficiency - focus on new architectures
experiment_configs = [
    # Current best configuration (ResNet baseline)
    {'architecture': 'resnet', 'learning_rate': 1e-3, 'batch_size': 32, 'weight_decay': 1e-5, 'dropout': 0.5},
    
    # NEW: SE-ResNet (ResNet with attention - typically +1-2% improvement)
    {'architecture': 'seresnet', 'learning_rate': 1e-3, 'batch_size': 32, 'weight_decay': 1e-5, 'dropout': 0.5},
    {'architecture': 'seresnet', 'learning_rate': 5e-4, 'batch_size': 32, 'weight_decay': 1e-4, 'dropout': 0.3},
    
    # NEW: DenseNet (excellent feature reuse - promising for small data)
    {'architecture': 'densenet', 'learning_rate': 1e-3, 'batch_size': 32, 'weight_decay': 1e-5, 'dropout': 0.5},
    {'architecture': 'densenet', 'learning_rate': 5e-4, 'batch_size': 32, 'weight_decay': 1e-4, 'dropout': 0.3},
    {'architecture': 'densenet', 'learning_rate': 1e-4, 'batch_size': 48, 'weight_decay': 1e-5, 'dropout': 0.5},
    
    # NEW: Inception (multi-scale feature extraction)
    {'architecture': 'inception', 'learning_rate': 1e-3, 'batch_size': 32, 'weight_decay': 1e-5, 'dropout': 0.5},
    {'architecture': 'inception', 'learning_rate': 5e-4, 'batch_size': 48, 'weight_decay': 1e-4, 'dropout': 0.3},
    
    # NEW: MobileNetV2 (efficient, depthwise separable convolutions)
    {'architecture': 'mobilenetv2', 'learning_rate': 1e-3, 'batch_size': 32, 'weight_decay': 1e-5, 'dropout': 0.5},
    {'architecture': 'mobilenetv2', 'learning_rate': 5e-4, 'batch_size': 48, 'weight_decay': 1e-4, 'dropout': 0.3},
    
    # SE-ResNet variations
    {'architecture': 'seresnet', 'learning_rate': 1e-4, 'batch_size': 16, 'weight_decay': 1e-5, 'dropout': 0.5},
    
    # Advanced CNN baseline
    {'architecture': 'advanced', 'learning_rate': 1e-3, 'batch_size': 32, 'weight_decay': 1e-4, 'dropout': 0.5},
    
    # Additional DenseNet with different LR
    {'architecture': 'densenet', 'learning_rate': 1e-3, 'batch_size': 48, 'weight_decay': 1e-4, 'dropout': 0.3},
]

print(f"Total experiments to run: {len(experiment_configs)}")
for i, config in enumerate(experiment_configs, 1):
    print(f"{i:2d}. {config['architecture']:10s} | LR={config['learning_rate']:.0e} | BS={config['batch_size']:2d} | WD={config['weight_decay']:.0e} | DO={config['dropout']:.1f}")

Total experiments to run: 13
 1. resnet     | LR=1e-03 | BS=32 | WD=1e-05 | DO=0.5
 2. seresnet   | LR=1e-03 | BS=32 | WD=1e-05 | DO=0.5
 3. seresnet   | LR=5e-04 | BS=32 | WD=1e-04 | DO=0.3
 4. densenet   | LR=1e-03 | BS=32 | WD=1e-05 | DO=0.5
 5. densenet   | LR=5e-04 | BS=32 | WD=1e-04 | DO=0.3
 6. densenet   | LR=1e-04 | BS=48 | WD=1e-05 | DO=0.5
 7. inception  | LR=1e-03 | BS=32 | WD=1e-05 | DO=0.5
 8. inception  | LR=5e-04 | BS=48 | WD=1e-04 | DO=0.3
 9. mobilenetv2 | LR=1e-03 | BS=32 | WD=1e-05 | DO=0.5
10. mobilenetv2 | LR=5e-04 | BS=48 | WD=1e-04 | DO=0.3
11. seresnet   | LR=1e-04 | BS=16 | WD=1e-05 | DO=0.5
12. advanced   | LR=1e-03 | BS=32 | WD=1e-04 | DO=0.5
13. densenet   | LR=1e-03 | BS=48 | WD=1e-04 | DO=0.3


## 3. Utility Functions for Architecture Variants

In [9]:
def create_model_for_config(config, device):
    """Create model with specified configuration."""
    model = get_model(config['architecture'], num_classes=7, device=device)
    return model

def count_parameters(model):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def prepare_dataloaders(config):
    """Create data loaders with specified batch size."""
    train_loader, val_loader = create_dataloaders(
        X_train, y_train,
        X_val, y_val,
        batch_size=config['batch_size'],
        device=device,
        augment=True
    )
    return train_loader, val_loader

# Test configuration creation
print("Testing configuration creation...")
test_config = experiment_configs[0]
test_model = create_model_for_config(test_config, device)
total_params, trainable_params = count_parameters(test_model)
print(f"✓ {test_config['architecture'].upper()} model: {total_params:,} total params, {trainable_params:,} trainable")

Testing configuration creation...
✓ RESNET model: 2,798,663 total params, 2,798,663 trainable


## 4. Build Experiment Tracking Framework

In [10]:
# Initialize experiment tracking
experiment_results = []
best_val_accuracy = 0
best_config = None

def run_experiment(config_id, config):
    """Run single experiment and record results."""
    global best_val_accuracy, best_config
    
    print(f"\n{'='*80}")
    print(f"Experiment {config_id}/{len(experiment_configs)} | {config['architecture'].upper()}")
    print(f"{'='*80}")
    print(f"Learning Rate: {config['learning_rate']:.0e}")
    print(f"Batch Size: {config['batch_size']}")
    print(f"Weight Decay: {config['weight_decay']:.0e}")
    print(f"Dropout: {config['dropout']:.1f}")
    
    try:
        # Start timer
        start_time = time()
        
        # Create model
        model = create_model_for_config(config, device)
        total_params, trainable_params = count_parameters(model)
        
        # Create dataloaders
        train_loader, val_loader = prepare_dataloaders(config)
        
        # Generate unique model name
        model_name = f"exp{config_id}_{config['architecture']}_lr{config['learning_rate']:.0e}_bs{config['batch_size']}"
        
        # Train model
        history = train_model(
            model,
            train_loader,
            val_loader,
            epochs=100,
            learning_rate=config['learning_rate'],
            device=device,
            model_name=model_name,
            class_weights=class_weights,
            weight_decay=config['weight_decay']
        )
        
        # Record results
        elapsed = time() - start_time
        best_val_acc = max(history['val_accuracy'])
        val_loss = min(history['val_loss'])
        
        result = {
            'exp_id': config_id,
            'architecture': config['architecture'],
            'learning_rate': config['learning_rate'],
            'batch_size': config['batch_size'],
            'weight_decay': config['weight_decay'],
            'dropout': config['dropout'],
            'total_params': total_params,
            'best_val_accuracy': best_val_acc,
            'best_val_loss': val_loss,
            'epochs_trained': len(history['train_loss']),
            'training_time_sec': elapsed,
            'history': history,
            'model_path': f'saved_models/{model_name}_best.pt'
        }
        
        experiment_results.append(result)
        
        # Update best model
        if best_val_acc > best_val_accuracy:
            best_val_accuracy = best_val_acc
            best_config = config_id
            print(f"✓ NEW BEST: Val Acc = {best_val_acc:.4f}")
        
        print(f"✓ Completed in {elapsed/60:.1f} min | Val Acc: {best_val_acc:.4f}")
        
        return result
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        return None

print("✓ Experiment framework ready")

✓ Experiment framework ready


## 5. Run Hyperparameter Search Experiments

In [11]:
# Clear GPU cache before starting
torch.cuda.empty_cache()

# Run all experiments
print(f"Starting optimization with {len(experiment_configs)} configurations...")
print(f"Started at: {pd.Timestamp.now()}\n")

successful_experiments = 0
for config_id, config in enumerate(experiment_configs, 1):
    result = run_experiment(config_id, config)
    if result is not None:
        successful_experiments += 1

print(f"\n{'='*80}")
print(f"EXPERIMENTS COMPLETED")
print(f"{'='*80}")
print(f"Total: {len(experiment_configs)} | Successful: {successful_experiments} | Failed: {len(experiment_configs) - successful_experiments}")
print(f"Finished at: {pd.Timestamp.now()}")

Starting optimization with 13 configurations...
Started at: 2026-04-09 14:31:22.260649


Experiment 1/13 | RESNET
Learning Rate: 1e-03
Batch Size: 32
Weight Decay: 1e-05
Dropout: 0.5

Training exp1_resnet_lr1e-03_bs32
Device: cuda
Epoch     Train Loss     Train Acc      Val Loss       Val Acc        Learning Rate       
------------------------------------------------------------------------------------------------------------------------



5         1.6655         0.3864         1.6059         0.3943         1.0e-03             


10        1.4033         0.4928         1.3733         0.5078         1.0e-03             


15        1.2701         0.5417         1.2603         0.5608         1.0e-03             


20        1.1783         0.5773         1.2005         0.5785         1.0e-03             


...

EarlyStopping counter: 5/15


65        1.1891         0.5556         1.3301         0.5526         6.3e-05             


EarlyStopping counter: 10/15


70        1.1712         0.5583         1.3092         0.5489         3.1e-05             


EarlyStopping counter: 15/15
Early stopping at epoch 73 (best: 58)

Training stopped at epoch 74

Best model saved to saved_models/exp13_densenet_lr1e-03_bs48_best.pt
Final model saved to saved_models/exp13_densenet_lr1e-03_bs48_final.pt
✓ Completed in 42.8 min | Val Acc: 0.5547

EXPERIMENTS COMPLETED
Total: 13 | Successful: 13 | Failed: 0
Finished at: 2026-04-09 22:58:21.516388


## 6. Compare and Analyze Results

In [12]:
# Create results DataFrame
results_df = pd.DataFrame([
    {
        'Exp': r['exp_id'],
        'Architecture': r['architecture'].upper(),
        'LR': f"{r['learning_rate']:.0e}",
        'BS': r['batch_size'],
        'WD': f"{r['weight_decay']:.0e}",
        'Val Acc': f"{r['best_val_accuracy']:.4f}",
        'Val Loss': f"{r['best_val_loss']:.4f}",
        'Epochs': r['epochs_trained'],
        'Time (min)': f"{r['training_time_sec']/60:.1f}",
        'Params': f"{r['total_params']/1e6:.1f}M"
    }
    for r in experiment_results
])

print("\n" + "="*120)
print("HYPERPARAMETER OPTIMIZATION RESULTS")
print("="*120)
print(results_df.to_string(index=False))
print("="*120)

# Find best configurations
best_by_val = experiment_results[0]
for r in experiment_results:
    if r['best_val_accuracy'] > best_by_val['best_val_accuracy']:
        best_by_val = r

print(f"\n✨ BEST MODEL:")
print(f"  Experiment: {best_by_val['exp_id']}")
print(f"  Architecture: {best_by_val['architecture'].upper()}")
print(f"  Learning Rate: {best_by_val['learning_rate']:.0e}")
print(f"  Batch Size: {best_by_val['batch_size']}")
print(f"  Weight Decay: {best_by_val['weight_decay']:.0e}")
print(f"  Validation Accuracy: {best_by_val['best_val_accuracy']:.4f}")
print(f"  Model Path: {best_by_val['model_path']}")


HYPERPARAMETER OPTIMIZATION RESULTS
 Exp Architecture    LR  BS    WD Val Acc Val Loss  Epochs Time (min) Params
   1       RESNET 1e-03  32 1e-05  0.6553   1.1289      49       40.0   2.8M
   2     SERESNET 1e-03  32 1e-05  0.6536   1.1187      42       39.9   2.8M
   3     SERESNET 5e-04  32 1e-04  0.6553   1.0825      56       50.4   2.8M
   4     DENSENET 1e-03  32 1e-05  0.6014   1.1729      70       47.5   0.1M
   5     DENSENET 5e-04  32 1e-04  0.5897   1.2213      73       46.9   0.1M
   6     DENSENET 1e-04  48 1e-05  0.5376   1.3123      79       48.9   0.1M
   7    INCEPTION 1e-03  32 1e-05  0.5603   1.3033      37       16.4   0.0M
   8    INCEPTION 5e-04  48 1e-04  0.5498   1.3009      80       28.3   0.0M
   9  MOBILENETV2 1e-03  32 1e-05  0.6336   1.1423      56       36.8   0.7M
  10  MOBILENETV2 5e-04  48 1e-04  0.6238   1.1380      55       35.2   0.7M
  11     SERESNET 1e-04  16 1e-05  0.6571   1.1004      45       40.0   2.8M
  12     ADVANCED 1e-03  32 1e-04  0.63

In [13]:
# Analyze hyperparameter impact
print("\n" + "="*80)
print("HYPERPARAMETER SENSITIVITY ANALYSIS")
print("="*80)

# By architecture
print("\n[By Architecture]")
for arch in set(r['architecture'] for r in experiment_results):
    arch_results = [r for r in experiment_results if r['architecture'] == arch]
    avg_acc = np.mean([r['best_val_accuracy'] for r in arch_results])
    max_acc = max(r['best_val_accuracy'] for r in arch_results)
    print(f"  {arch.upper():10s}: Avg={avg_acc:.4f}, Max={max_acc:.4f} ({len(arch_results)} configs)")

# By learning rate
print("\n[By Learning Rate]")
for lr in sorted(set(r['learning_rate'] for r in experiment_results)):
    lr_results = [r for r in experiment_results if r['learning_rate'] == lr]
    avg_acc = np.mean([r['best_val_accuracy'] for r in lr_results])
    max_acc = max(r['best_val_accuracy'] for r in lr_results)
    print(f"  {lr:.0e}: Avg={avg_acc:.4f}, Max={max_acc:.4f} ({len(lr_results)} configs)")

# By batch size
print("\n[By Batch Size]")
for bs in sorted(set(r['batch_size'] for r in experiment_results)):
    bs_results = [r for r in experiment_results if r['batch_size'] == bs]
    avg_acc = np.mean([r['best_val_accuracy'] for r in bs_results])
    max_acc = max(r['best_val_accuracy'] for r in bs_results)
    print(f"  {bs:2d}: Avg={avg_acc:.4f}, Max={max_acc:.4f} ({len(bs_results)} configs)")


HYPERPARAMETER SENSITIVITY ANALYSIS

[By Architecture]
  ADVANCED  : Avg=0.6385, Max=0.6385 (1 configs)
  MOBILENETV2: Avg=0.6287, Max=0.6336 (2 configs)
  DENSENET  : Avg=0.5708, Max=0.6014 (4 configs)
  INCEPTION : Avg=0.5550, Max=0.5603 (2 configs)
  RESNET    : Avg=0.6553, Max=0.6553 (1 configs)
  SERESNET  : Avg=0.6553, Max=0.6571 (3 configs)

[By Learning Rate]
  1e-04: Avg=0.5974, Max=0.6571 (2 configs)
  5e-04: Avg=0.6047, Max=0.6553 (4 configs)
  1e-03: Avg=0.6139, Max=0.6553 (7 configs)

[By Batch Size]
  16: Avg=0.6571, Max=0.6571 (1 configs)
  32: Avg=0.6235, Max=0.6553 (8 configs)
  48: Avg=0.5665, Max=0.6238 (4 configs)


In [ ]:
# Visualization: comparison of the best 4 models by validation accuracy
if len(experiment_results) == 0:
    print("No experiment results found. Run the experiments first.")
else:
    top_k = min(4, len(experiment_results))
    top_models = sorted(experiment_results, key=lambda x: x['best_val_accuracy'], reverse=True)[:top_k]

    top_df = pd.DataFrame([
        {
            "Label": f"Exp {r['exp_id']} ({r['architecture'].upper()})",
            "Val Accuracy": r["best_val_accuracy"],
            "Val Loss": r["best_val_loss"],
            "Params (M)": r["total_params"] / 1e6,
            "Time (min)": r["training_time_sec"] / 60
        }
        for r in top_models
    ])

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Top {top_k} Model Comparison", fontsize=14, fontweight="bold")

    # 1) Validation Accuracy
    sns.barplot(data=top_df, x="Val Accuracy", y="Label", palette="viridis", ax=axes[0, 0])
    axes[0, 0].set_title("Best Validation Accuracy")
    axes[0, 0].set_xlim(0, max(0.7, top_df["Val Accuracy"].max() + 0.05))

    # 2) Validation Loss
    sns.barplot(data=top_df, x="Val Loss", y="Label", palette="magma", ax=axes[0, 1])
    axes[0, 1].set_title("Best Validation Loss")

    # 3) Parameters
    sns.barplot(data=top_df, x="Params (M)", y="Label", palette="Blues", ax=axes[1, 0])
    axes[1, 0].set_title("Model Size (Million Parameters)")

    # 4) Training Time
    sns.barplot(data=top_df, x="Time (min)", y="Label", palette="Oranges", ax=axes[1, 1])
    axes[1, 1].set_title("Training Time (minutes)")

    plt.tight_layout()
    plt.show()

    # Optional: validation accuracy curves of top models
    plt.figure(figsize=(10, 6))
    for r in top_models:
        plt.plot(r["history"]["val_accuracy"], label=f"Exp {r['exp_id']} - {r['architecture'].upper()}")
    plt.title(f"Validation Accuracy Curves (Top {top_k} Models)")
    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()